In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
import torch
import torch.nn.functional as F


class DiffusionQuotaHelper(ABC):
    @abstractmethod
    def get_quota(self, step_current):
        pass
    # end
# end

class BlockDiffusionQuotaHelper(DiffusionQuotaHelper):
    def __init__(self, block_mask_index: torch.Tensor, steps_per_block: int) -> torch.Tensor:
        device = block_mask_index.device
        dtype = torch.long

        total = block_mask_index.sum(dim=1)                  # (B,)
        base  = torch.div(total, steps_per_block, rounding_mode='floor')  # (B,)
        rem   = total - base * steps_per_block                         # (B,)

        # Start with base for all steps
        num_transfer_tokens = base.unsqueeze(1).expand(-1, steps_per_block).to(dtype)  # (B, steps)

        # Add +1 to the first `rem[b]` steps for each batch b — without tensor slicing
        cols = torch.arange(steps_per_block, device=device).unsqueeze(0)               # (1, steps)
        add_mask = cols < rem.unsqueeze(1)                                   # (B, steps)
        self.num_transfer_tokens = num_transfer_tokens + add_mask.to(dtype)       # (B, steps)
    # end

    def get_quota(self, step_current):
        quota_current = self.num_transfer_tokens[:, step_current]

        if quota_current.dim() == 2 and quota_current.size(1) == 1:
            quota_current = quota_current.squeeze(1)
        # end

        return quota_current
    # end
# end



class ConfKSorter:

    def argsort(self, conf_all):
        idx_sorted = torch.argsort(conf_all, dim=1, descending=True)
        return idx_sorted
    # end
# end

class RandomKSorter(ConfKSorter):
    def argsort(self, confidence, snapshot):

        confidence = torch.where(
                snapshot.mask_mask,
                torch.rand(confidence.shape[0], confidence.shape[1], device=confidence.device),
                confidence
            )

        return self.super(confidence)

    # end
# end


class TopKSorter(ConfKSorter):
    def argsort(self, confidence, snapshot):
        return self.super(confidence)
    # end
# end



class ConfCollectorInterface(ABC):
    @abstractmethod
    def get_index(self, snapshot):
        pass
    # end
# end

class TruthCollector(ConfCollectorInterface):
    def get_index(self, snapshot):
        index = snapshot.y.unsqueeze(-1)
        return index
    # end
# end

class MaxCollector(ConfCollectorInterface):
    def get_index(self, snapshot):
        index = snapshot.x0
        return index
    # end
# end

class LogitsTransformer:
    def transform_logits(self, logits, collector):
        p = F.softmax(logits.to(torch.float64), dim=-1)
        x0_p = collector.gather_x0_p(p, self)
        return x0_p
    # end
# end

class SimpleLogitsSnapshot:

    def __init__(self, logits, x, y, id_mask):
        self.id_mask = id_mask

        self.logits = logits
        self.x = x
        self.y = y

        self.x0 = torch.argmax(self.logits, dim=-1)

        self.p_finalized = torch.zeros(self.logits.shape, dtype=torch.float64).to(self.x.device)
    # end

    def transform_logits(self, collector):

        logits_tranform = self.logits
        p = F.softmax(logits_tranform.to(torch.float64), dim=-1)

        index_p_all = collector.get_index(self)
        x0_p = torch.gather(p, dim=-1, index=index_p_all)
        neg_inf = torch.tensor(torch.finfo(x0_p.dtype).min, device=x0_p.device, dtype=x0_p.dtype)

        mask_mask = self.x == self.id_mask
        conf = torch.where(mask_mask, x0_p, neg_inf)  # (B, L)   # so only the masked part has confidence

        return conf
    # end
    def materilize_by_idx_(self, idx, conf):
        x0 = torch.gather(self.x0, dim=-1, index=idx)

        self.x.scatter_(1, x0, idx)
        self.p_finalized.scatter_(1, idx, conf)
    # end


    def update_logits_(self, logits, idx_logits):
        self.logits.scatter_(1, logits, idx_logits)
        x0 = torch.argmax(logits, dim=-1)
        self.x0.scatter_(1, x0, idx_logits)
    # end
# end


@dataclass
class DiffusionConfig:
    num_blocks: int
    step_per_block: int
    size_block: int
    id_mask: int
    len_prompt: int
    klass_sorter: ConfKSorter
    klass_collector: ConfCollectorInterface
# end


In [ ]:
model = LlaDAModel()
model.enable_past_kv()

# 处理past_key_values
def run_model_semi_cache_snapshot(model, x, y, config_diffusion):

    num_blocks = config_diffusion.num_blocks
    step_per_block = config_diffusion.step_per_block
    size_block = config_diffusion.size_block
    id_mask = config_diffusion.id_mask
    len_prompt = config_diffusion.len_prompt
    sorter = config_diffusion.sorter
    collector = config_diffusion.collector

    idx_refresh = torch.tensor([[]])
    p_final = torch.zeros(x.shape, dtype=torch.float64, device=x.device)

    for id_block in range(num_blocks):
        position_start = len_prompt + id_block * size_block
        position_end = position_start + size_block
        mask_mask_blk = x[:,:position_end] == id_mask

        quota_helper = BlockDiffusionQuotaHelper(mask_mask_blk, size_block)

        idx_denoising = x[:, position_start:position_end]
        idx_current = torch.cat([idx_refresh, idx_denoising], dim=-1) 
        logits = model(x[:, idx_current], idx_current=idx_current).logits

        snapshot = SimpleLogitsSnapshot(logits, x, y, id_mask)
        conf_snapshot = snapshot.transform_logits(collector)

        for step in range(step_per_block):

            '''固定的重排拿一个'''
            idx_sorted_by_conf = sorter.argsort(conf_snapshot, snapshot)    # 重全排    这里又有问题啦
            num_unmask = quota_helper.get_quota(step)
            idx_denoising = idx_sorted_by_conf[:, :num_unmask]
            idx_current = torch.cat([idx_refresh, idx_denoising], dim=-1)   # 原本的位置
            logits = model(x[:, idx_current], idx_current=idx_current).logits

            logits_denoising = logits[:, idx_denoising]

            '''固定的materilize step'''
            snapshot.update_logits_(logits_denoising, idx_denoising)
            conf_snapshot = snapshot.transform_logits(collector) # conf recal???
            snapshot.materilize_by_idx_(idx_denoising, conf_snapshot[:, idx_denoising])
            idx_refresh = idx_denoising
        # end

        p_final.scatter_(1, idx_current, snapshot.get_p_finalized())
    # end

    return torch.gather(p_final, dim=-1, index=torch.arange(len_prompt, x.shape[-1]))
# end

In [ ]:
model = LlaDAModel()
model.enable_past_kv()

# 处理past_key_values
def run_model_semi_cache(model, x, y, config_diffusion):

    num_blocks = config_diffusion.num_blocks
    step_per_block = config_diffusion.step_per_block
    size_block = config_diffusion.size_block
    id_mask = config_diffusion.id_mask
    len_prompt = config_diffusion.len_prompt
    sorter = config_diffusion.sorter
    collector = config_diffusion.collector

    p_final = torch.zeros(x.shape, dtype=torch.float64, device=x.device)

    idx_denoising = x[:, :len_prompt]
    model(x[:, idx_denoising], idx_current=idx_denoising)   # save prompt for once

    for id_block in range(num_blocks):
        position_start = len_prompt + id_block * size_block
        position_end = position_start + size_block
        mask_mask_blk = x[:,:position_end] == id_mask

        B = x.shape[0]
        L = position_end - position_start
        
        idx_denoising = torch.arange(position_start, position_end).unsqueeze(0).expand(B, L)
        quota_helper = BlockDiffusionQuotaHelper(mask_mask_blk, size_block)

        for step in range(step_per_block):
            x_denoising,  y_denoising= x[:, idx_denoising], y[:, idx_denoising]
            logits = model(x_denoising, idx_current=idx_denoising).logits

            snapshot = SimpleLogitsSnapshot(logits, x_denoising, y_denoising, id_mask, previous=snapshot)
            conf_snapshot = snapshot.transform_logits(collector)

            idx_sorted_by_conf = sorter.argsort(conf_snapshot, snapshot)
            num_unmask = quota_helper.get_quota(step)
            idx_transform = idx_sorted_by_conf[:, :num_unmask]

            snapshot.materilize_by_idx_(idx_transform, conf_snapshot[:, idx_transform])
        # end for step

        p_final.scatter_(1, idx_denoising, snapshot.get_p_finalized())
    # end for block

    return torch.gather(p_final, dim=-1, index=torch.arange(len_prompt, x.shape[-1]))
# end

In [ ]:
model = LlaDAModel()

# 处理past_key_values
def run_model_semi(model, x, y, config_diffusion):

    num_blocks = config_diffusion.num_blocks
    step_per_block = config_diffusion.step_per_block
    size_block = config_diffusion.size_block
    id_mask = config_diffusion.id_mask
    len_prompt = config_diffusion.len_prompt
    sorter = config_diffusion.sorter
    collector = config_diffusion.collector
    
    p_final = torch.zeros(x.shape, dtype=torch.float64, device=x.device)
    position_start = 0

    for id_block in range(num_blocks):
        position_end = position_start + id_block * size_block
        mask_mask_blk = x[:,:position_end] == id_mask

        B = x.shape[0]
        L = position_end - position_start
        
        idx_denoising = torch.arange(position_start, position_end).unsqueeze(0).expand(B, L)
        quota_helper = BlockDiffusionQuotaHelper(mask_mask_blk, size_block)

        for step in range(step_per_block):
            logits = model(x[:, idx_denoising], idx_current=idx_denoising).logits

            snapshot = SimpleLogitsSnapshot(logits, x, y, id_mask)
            conf_snapshot = snapshot.transform_logits(collector)

            idx_sorted_by_conf = sorter.argsort(conf_snapshot, snapshot)
            num_unmask = quota_helper.get_quota(step)
            idx_transform = idx_sorted_by_conf[:, :num_unmask]

            snapshot.materilize_by_idx_(idx_transform, conf_snapshot[:, idx_transform])
            p_final.scatter_(1, idx_transform, snapshot.get_p_finalized())
        # end for step

    # end for block

    return torch.gather(p_final, dim=-1, index=torch.arange(len_prompt, x.shape[-1]))
# end

In [ ]:
def run_model_semi_cache_refresh(model, x, y, config_diffusion):
    
    num_blocks = config_diffusion.num_blocks
    step_per_block = config_diffusion.step_per_block
    size_block = config_diffusion.size_block
    id_mask = config_diffusion.id_mask
    len_prompt = config_diffusion.len_prompt
    sorter = config_diffusion.sorter
    collector = config_diffusion.collector
    helper_refresh = config_diffusion.helper_refresh

    p_final = torch.zeros(x.shape, dtype=torch.float64, device=x.device)
    idx_denoising = x[:, :len_prompt]
    model(x[:, idx_denoising], idx_current=idx_denoising)   # save prompt for once

    for id_block in range(num_blocks):
        position_start = len_prompt + id_block * size_block
        position_end = position_start + size_block
        mask_mask_blk = x[:,:position_end] == id_mask
        quota_helper = BlockDiffusionQuotaHelper(mask_mask_blk, size_block)

        B = x.shape[0]
        L = position_end - position_start
        
        idx_denoising = torch.arange(position_start, position_end).unsqueeze(0).expand(B, L)

        for step in range(step_per_block):
            idx_refresh = helper_refresh.get_refresh_idx(x,
                            position_start=position_start,
                            step_in_block=step,
                            id_block=id_block,
                            return_sorted=True
                        )
            idx_current = torch.cat([idx_refresh, idx_denoising])
            logits = model(x[:, idx_current], idx_current=idx_current).logits

            snapshot = SimpleLogitsSnapshot(logits, x, y, id_mask)
            conf_snapshot = snapshot.transform_logits(collector)

            idx_sorted_by_conf = sorter.argsort(conf_snapshot, snapshot)
            num_unmask = quota_helper.get_quota(step)
            idx_transform = idx_sorted_by_conf[:, :num_unmask]

            snapshot.materilize_by_idx_(idx_transform, conf_snapshot[:, idx_transform])
        # end
    # end
# end